In [1]:
import ee
import pandas as pd

ee.Initialize(project="ee-chowdharypranav2005")

districts = pd.read_csv(
    "../data/raw/outbreaks/district_master.csv"
)

print("Loaded", len(districts), "districts")

Loaded 9 districts


In [2]:
elevation = ee.Image("USGS/SRTMGL1_003")

In [3]:
def extract_elevation(lat, lon, buffer_radius=50000):

    region = ee.Geometry.Point([lon, lat]).buffer(buffer_radius)

    stats = elevation.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=30,
        maxPixels=1e9
    )

    return stats.get("elevation").getInfo()

In [4]:
value = extract_elevation(
    lat=13.9299,
    lon=75.5681
)

print(value)

677.877598988984


In [5]:
records = []

for _, district in districts.iterrows():

    elevation_value = extract_elevation(
        district["Latitude"],
        district["Longitude"]
    )

    records.append({
        "State": district["State"],
        "District": district["District"],
        "Elevation": elevation_value
    })

elevation_df = pd.DataFrame(records)

elevation_df.head()

,State,District,Elevation
0,Karnataka,Shivamogga,677.877599
1,Karnataka,Uttara Kannada,429.982021
2,Karnataka,Chikkamagaluru,890.594716
3,Karnataka,Kodagu,682.786387
4,Kerala,Wayanad,651.723530


In [6]:
elevation_df.to_csv(
    "../data/processed/elevation_western_ghats.csv",
    index=False
)

print("Elevation dataset saved!")

Elevation dataset saved!


In [7]:
print(elevation_df.describe())

        Elevation
count    9.000000
mean   427.189948
std    309.026662
min     71.222575
25%    145.369705
50%    429.982021
75%    677.877599
max    890.594716
